# LeetCode 420 — Strong Password Checker (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def strongPasswordChecker(self, password: str) -> int:

```

Đề bài nghe rất "đời thường": kiểm tra một mật khẩu có **mạnh** không, nếu chưa thì cần **ít nhất bao nhiêu bước sửa** để nó mạnh. Nhưng bài này nổi tiếng là một trong những bài Hard "khó chịu" nhất LeetCode, vì ba điều kiện của mật khẩu **đan xen** vào nhau: một bước sửa có thể giải quyết cùng lúc hai vấn đề, hoặc giải quyết vấn đề này nhưng làm hỏng vấn đề kia. Bài không cần cấu trúc dữ liệu phức tạp — nó cần **phân tích trường hợp** (case analysis) thật cẩn thận và một quan sát rất đẹp về **phép chia lấy dư cho 3**.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc đề thật kỹ: ba điều kiện của mật khẩu mạnh và ba thao tác sửa (chèn / xóa / thay).
2. Các công cụ Python nền tảng: `str.islower()`, `isupper()`, `isdigit()`, `any(...)`, tách chuỗi thành các **đoạn lặp** (run), phép `//` và `%` với số 3, hàng đợi `deque` và **BFS**.
3. **Cách 0 — BFS trên chuỗi** (vét cạn theo số bước): chậm, chỉ chạy được với mật khẩu rất ngắn, nhưng **chắc chắn đúng** → dùng làm "trọng tài" kiểm tra các cách sau.
4. **Bước đột phá**: nhìn bài toán như ba "món nợ" (`missing`, độ dài, đoạn lặp), hiểu vì sao một đoạn lặp dài `L` cần đúng `L // 3` lần thay.
5. **Cách 1 — Phân tích ba trường hợp độ dài + xóa tham lam từng ký tự** (luôn xóa ở đoạn có `L % 3` nhỏ nhất): lời giải chính, kèm trace từng bước.
6. **Cách 2 — Mảng đếm theo mod 3** (`cnt[0]`, `cnt[1]`, `cnt[2]`): tính thẳng bằng công thức, `O(n)` thời gian, `O(1)` bộ nhớ — bản nộp LeetCode.
7. **Trọng tài DP** cho mật khẩu dài hơn 20 ký tự (khi BFS bó tay), kèm kiến thức **bitmask**.
8. Vì sao lời giải đúng (cận dưới + lập luận tham lam), độ phức tạp, lỗi hay gặp, mẹo nhận dạng, biến thể và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Một mật khẩu được coi là **mạnh** (strong) nếu thỏa **tất cả** các điều kiện sau:

- Có **ít nhất 6** ký tự và **nhiều nhất 20** ký tự.
- Chứa **ít nhất một chữ thường** (lowercase letter), **ít nhất một chữ hoa** (uppercase letter) và **ít nhất một chữ số** (digit).
- **Không** chứa ba ký tự giống nhau đứng liền nhau (ví dụ `"B`**`aaa`**`bb0"` là yếu, nhưng `"Baaba0"` là mạnh).

Cho chuỗi `password`, trả về **số bước tối thiểu** để biến `password` thành mật khẩu mạnh. Nếu `password` đã mạnh thì trả về `0`.

Trong **một bước**, bạn được làm **một** trong ba việc:

- **Chèn** (insert) một ký tự vào `password`,
- **Xóa** (delete) một ký tự khỏi `password`, hoặc
- **Thay** (replace) một ký tự của `password` bằng một ký tự khác.

**Example 1**

```text
Input:  password = "a"
Output: 5
```

**Example 2**

```text
Input:  password = "aA1"
Output: 3
```

**Example 3**

```text
Input:  password = "1337C0d3"
Output: 0
```

**Constraints**

- `1 <= password.length <= 50`
- `password` chỉ gồm chữ cái (letters), chữ số (digits), dấu chấm `'.'` hoặc dấu chấm than `'!'`.

## 2. Đọc đề thật kỹ trước khi code

Hãy "dịch" đề thành những câu hỏi cụ thể:

| Điều kiện | Câu hỏi kiểm tra | Ví dụ vi phạm |
|---|---|---|
| Độ dài | `6 <= len(password) <= 20`? | `"aA1"` (quá ngắn), chuỗi 25 ký tự (quá dài) |
| Đủ 3 loại ký tự | có chữ thường? có chữ hoa? có chữ số? | `"abcdef"` thiếu chữ hoa và chữ số |
| Không lặp 3 | có vị trí `i` nào mà `p[i] == p[i+1] == p[i+2]`? | `"Baaabb0"` có `aaa` |

Vài điều rất dễ bỏ sót:

- **Dấu `.` và `!` hợp lệ trong input nhưng không thuộc loại nào** trong ba loại cần có. Chuỗi `"......"` dài đúng 6 nhưng thiếu cả ba loại **và** có đoạn lặp dài 6.
- Ba điều kiện **ảnh hưởng lẫn nhau**. Ví dụ `"aaaaaa"`: thay ký tự thứ 3 thành `A` và ký tự thứ 6 thành `1` → `"aaAaa1"`. Chỉ **2 bước** mà giải quyết **cả** đoạn lặp **lẫn** hai loại ký tự còn thiếu.
- Ta chỉ cần trả về **số bước**, không cần trả về mật khẩu mới. Nhưng để hiểu, ta sẽ thường tự xây mật khẩu mới để thấy các bước "có thật".
- Constraints cho `n <= 50`, nên mật khẩu có thể **dài hơn 20** — khi đó bắt buộc phải xóa, và đây chính là phần khó nhất của bài.

## 3. Kiến thức nền 1 — Kiểm tra loại ký tự: `islower`, `isupper`, `isdigit`, `any`

Chuỗi (`str`) trong Python có sẵn các phương thức hỏi "ký tự này thuộc loại gì":

- `c.islower()` → `True` nếu `c` là chữ thường (`'a'..'z'`).
- `c.isupper()` → `True` nếu `c` là chữ hoa (`'A'..'Z'`).
- `c.isdigit()` → `True` nếu `c` là chữ số (`'0'..'9'`).

Hàm có sẵn `any(...)` nhận một dãy giá trị đúng/sai và trả về `True` nếu **có ít nhất một** giá trị `True`. Viết `any(c.isdigit() for c in password)` đọc là: "có ký tự nào trong `password` là chữ số không?".

Một mẹo nhỏ: trong Python `True` cộng được như số `1`, `False` như số `0`. Nên `3 - (có_thường + có_hoa + có_số)` chính là **số loại còn thiếu** — ta gọi là `missing`.

In [1]:
for c in ["a", "Z", "7", ".", "!"]:
    print(repr(c), "lower:", c.islower(), "| upper:", c.isupper(), "| digit:", c.isdigit())

print()
print("True + True + False =", True + True + False)
print("any([False, False, True]) =", any([False, False, True]))
print("Có chữ số trong 'abc1'?", any(c.isdigit() for c in "abc1"))


def count_missing(password: str) -> int:
    has_lower = any(c.islower() for c in password)
    has_upper = any(c.isupper() for c in password)
    has_digit = any(c.isdigit() for c in password)
    return 3 - (has_lower + has_upper + has_digit)

for p in ["a", "aA1", "1337C0d3", "......", "abcDEF", "!!!9"]:
    print(f"{p!r:12} missing = {count_missing(p)}")

'a' lower: True | upper: False | digit: False
'Z' lower: False | upper: True | digit: False
'7' lower: False | upper: False | digit: True
'.' lower: False | upper: False | digit: False
'!' lower: False | upper: False | digit: False

True + True + False = 2
any([False, False, True]) = True
Có chữ số trong 'abc1'? True
'a'          missing = 2
'aA1'        missing = 0
'1337C0d3'   missing = 0
'......'     missing = 3
'abcDEF'     missing = 1
'!!!9'       missing = 2


## 4. Kiến thức nền 2 — Kiểm tra "ba ký tự lặp liên tiếp" và hàm `is_strong`

Để biết có ba ký tự giống nhau liền nhau không, ta nhìn từng **cửa sổ** gồm 3 vị trí `i, i+1, i+2`.

Vì phải đọc tới `p[i + 2]`, chỉ số `i` chỉ được chạy tới `len(p) - 3`, tức là `range(len(p) - 2)`. Nếu chạy `range(len(p))` thì khi `i` ở gần cuối, `p[i + 2]` sẽ **vượt chỉ số** (IndexError).

Ví dụ `p = "Baaabb0"` (dài 7) → `i` chạy `0..4`:

| i | p[i] p[i+1] p[i+2] | Ba ký tự giống nhau? |
|---:|:---:|:---:|
| 0 | `B a a` | không |
| 1 | `a a a` | **có** → yếu |

Ghép cả ba điều kiện lại, ta được hàm `is_strong(p)` — đây là **định nghĩa** của "mật khẩu mạnh" mà mọi cách giải sau đều phải tôn trọng.

In [2]:
def has_triple(p: str) -> bool:
    for i in range(len(p) - 2):
        if p[i] == p[i + 1] == p[i + 2]:
            return True
    return False


def is_strong(p: str) -> bool:
    return 6 <= len(p) <= 20 and count_missing(p) == 0 and not has_triple(p)


for p in ["Baaabb0", "Baaba0", "a", "aA1", "1337C0d3", "aaAaa1"]:
    print(f"{p!r:12} triple={has_triple(p)!s:5}  strong={is_strong(p)}")

'Baaabb0'    triple=True   strong=False
'Baaba0'     triple=False  strong=True
'a'          triple=False  strong=False
'aA1'        triple=False  strong=False
'1337C0d3'   triple=False  strong=True
'aaAaa1'     triple=False  strong=True


## 5. Kiến thức nền 3 — Tách chuỗi thành các **đoạn lặp** (run)

Một **đoạn lặp** (run) là một khúc liên tiếp gồm cùng một ký tự, dài nhất có thể. Ví dụ:

```text
"aaaBB1111c"  →  "aaa" | "BB" | "1111" | "c"   → độ dài 3, 2, 4, 1
```

Điều kiện "không có 3 ký tự lặp" tương đương với: **mọi đoạn lặp đều dài tối đa 2**. Vì vậy toàn bộ bài toán chỉ quan tâm tới **độ dài các đoạn lặp**, không quan tâm chúng là ký tự gì.

Cách tách thủ công bằng **hai con trỏ** (two pointers) `i` và `j`:

1. `i` đứng ở đầu một đoạn.
2. `j` chạy từ `i` sang phải chừng nào `p[j] == p[i]`.
3. Khi `j` dừng, đoạn là `p[i:j]`, dài `j - i`. Đặt `i = j` và lặp lại.

Mỗi ký tự được `j` đi qua đúng một lần → `O(n)`.

Python còn có sẵn `itertools.groupby`, gom các phần tử **liền nhau bằng nhau** thành nhóm. Trong cell dưới, ta chỉ dùng nó để đối chiếu cho chắc; bản nộp LeetCode sẽ dùng hai con trỏ (dễ giải thích, không phụ thuộc thư viện).

In [3]:
def get_runs(p: str):
    runs = []                     # danh sách cặp (ký tự, độ dài)
    i = 0
    while i < len(p):
        j = i
        while j < len(p) and p[j] == p[i]:
            j += 1
        runs.append((p[i], j - i))
        i = j
    return runs

print(get_runs("aaaBB1111c"))
print(get_runs("Baaabb0"))
print(get_runs("......"))


# Đối chiếu với itertools.groupby (gom các phần tử liền nhau bằng nhau thành nhóm)
from itertools import groupby

p = "aaaBB1111c"
print([(ch, len(list(group))) for ch, group in groupby(p)])
print("Giống get_runs?", [(ch, len(list(g))) for ch, g in groupby(p)] == get_runs(p))

[('a', 3), ('B', 2), ('1', 4), ('c', 1)]
[('B', 1), ('a', 3), ('b', 2), ('0', 1)]
[('.', 6)]
[('a', 3), ('B', 2), ('1', 4), ('c', 1)]
Giống get_runs? True


## 6. Kiến thức nền 4 — Phép `//` và `%` với số 3

Sau này ta sẽ thấy: một đoạn lặp dài `L` cần **`L // 3`** lần thay, và **`L % 3`** quyết định đoạn đó "dễ rút gọn" tới mức nào khi phải xóa ký tự.

- `L // 3` là **chia lấy phần nguyên**: `7 // 3 = 2`.
- `L % 3` là **chia lấy dư**: `7 % 3 = 1` (vì `7 = 3*2 + 1`).
- `L % 3` chỉ có thể là `0`, `1` hoặc `2`.

Bảng dưới đây là "bảng cửu chương" của bài này — hãy để ý cột `(L-1) // 3`: khi bớt 1 ký tự, `L // 3` **chỉ giảm khi `L % 3 == 0`**.

In [4]:
print(" L | L//3 | L%3 | (L-1)//3 | bớt 1 ký tự có giảm L//3?")
for L in range(3, 13):
    print(f"{L:2} | {L//3:4} | {L%3:3} | {(L-1)//3:8} | {'CÓ' if (L-1)//3 < L//3 else 'không'}")

 L | L//3 | L%3 | (L-1)//3 | bớt 1 ký tự có giảm L//3?
 3 |    1 |   0 |        0 | CÓ
 4 |    1 |   1 |        1 | không
 5 |    1 |   2 |        1 | không
 6 |    2 |   0 |        1 | CÓ
 7 |    2 |   1 |        2 | không
 8 |    2 |   2 |        2 | không
 9 |    3 |   0 |        2 | CÓ
10 |    3 |   1 |        3 | không
11 |    3 |   2 |        3 | không
12 |    4 |   0 |        3 | CÓ


## 7. Kiến thức nền 5 — Hàng đợi `deque` và tìm kiếm theo chiều rộng (BFS)

Câu hỏi "ít nhất bao nhiêu bước" thường có một lời giải vét cạn tự nhiên: **BFS** (Breadth-First Search — tìm kiếm theo chiều rộng).

Hình dung mỗi **trạng thái** (ở đây: một chuỗi) là một **đỉnh**, mỗi thao tác là một **cạnh** đi sang trạng thái mới. BFS đi theo từng "lớp":

- Lớp 0: trạng thái ban đầu.
- Lớp 1: mọi trạng thái đi được bằng 1 bước.
- Lớp 2: mọi trạng thái mới đi được bằng 2 bước, ...

Trạng thái đích **đầu tiên** BFS gặp chắc chắn có số bước nhỏ nhất, vì BFS xét hết các trạng thái `k` bước rồi mới tới `k + 1` bước.

Để đi theo lớp, ta dùng **hàng đợi** (queue): vào sau thì ra sau (FIFO). Trong Python dùng `collections.deque`: `append(x)` thêm vào cuối, `popleft()` lấy ra ở đầu — cả hai đều `O(1)`. Ta cũng cần một `dict` `dist` ghi "trạng thái này đã gặp ở bước mấy" để không xét lại.

In [5]:
from collections import deque

q = deque()
q.append("A"); q.append("B"); q.append("C")
print("Hàng đợi:", q)
print("popleft ->", q.popleft(), "| còn lại:", q)
q.append("D")
print("Thêm D  ->", q)

Hàng đợi: deque(['A', 'B', 'C'])
popleft -> A | còn lại: deque(['B', 'C'])
Thêm D  -> deque(['B', 'C', 'D'])


Ví dụ mini trước khi áp vào mật khẩu: từ số `2`, mỗi bước được **cộng 1** hoặc **nhân 2**. Cần ít nhất mấy bước để tới `9`?

Đoán tay: `2 → 4 → 8 → 9` là 3 bước. BFS sẽ xác nhận không có cách nào 2 bước.

In [6]:
def min_steps_number(start, target):
    dist = {start: 0}
    queue = deque([start])
    while queue:
        x = queue.popleft()
        if x == target:
            return dist[x]
        for y in (x + 1, x * 2):
            if y <= target and y not in dist:   # vượt target thì không quay lại được
                dist[y] = dist[x] + 1
                queue.append(y)

print("2 -> 9 :", min_steps_number(2, 9), "bước")
print("3 -> 10:", min_steps_number(3, 10), "bước")   # 3 -> 4 -> 5 -> 10

2 -> 9 : 3 bước
3 -> 10: 3 bước


## 8. Cách 0 — BFS trên chuỗi (vét cạn, làm "trọng tài")

Áp đúng khuôn BFS vừa học:

- **Trạng thái** = một chuỗi.
- **Cạnh** = một thao tác: xóa `s[i]`, thay `s[i]` bằng ký tự `c`, hoặc chèn ký tự `c` vào vị trí `i`.
- **Đích** = chuỗi thỏa `is_strong`.

Vấn đề duy nhất: chèn/thay bằng ký tự **nào**? Có tới 64 ký tự hợp lệ, thử hết thì BFS nổ tung. May mắn là ta chỉ cần **3 ký tự `a`, `A`, `1`**. Lý do (đọc chậm):

1. Một ký tự **mới** (được chèn hoặc thay vào) chỉ quan trọng ở hai chỗ: nó thuộc **loại** nào (thường/hoa/số), và nó có **trùng với hàng xóm** không. Trùng hàng xóm thì chỉ làm đoạn lặp dài thêm — không bao giờ có lợi.
2. Với mỗi loại còn thiếu, ta chỉ cần **một** ký tự mới đại diện: `a`, `A` hoặc `1`. Ký tự gốc không thuộc loại đang thiếu, nên đại diện đó chắc chắn khác hàng xóm gốc.
3. Các ký tự mới còn lại chỉ cần **khác hai hàng xóm**. Có 3 lựa chọn `a`, `A`, `1` mà chỉ có 2 hàng xóm → luôn chọn được.

Vậy nếu có cách sửa tối ưu dùng ký tự bất kỳ, ta luôn đổi được nó thành cách sửa **cùng số bước** chỉ dùng `a`, `A`, `1`. BFS với 3 ký tự này vẫn cho **đáp án chính xác** — chỉ là nó chậm.

In [7]:
NEW_CHARS = "aA1"

def neighbors(s: str):
    result = []
    for i in range(len(s)):
        result.append(s[:i] + s[i + 1:])                 # xóa s[i]
        for c in NEW_CHARS:
            if c != s[i]:
                result.append(s[:i] + c + s[i + 1:])     # thay s[i] bằng c
    for i in range(len(s) + 1):
        for c in NEW_CHARS:
            result.append(s[:i] + c + s[i:])             # chèn c trước vị trí i
    return result

print("Một vài hàng xóm của 'aA':", neighbors("aA")[:8], "...")
for L in [5, 10, 20, 50]:
    print(f"Chuỗi dài {L:2} có khoảng {len(neighbors('.' * L)):3} hàng xóm (1 bước)")

Một vài hàng xóm của 'aA': ['A', 'AA', '1A', 'a', 'aa', 'a1', 'aaA', 'AaA'] ...
Chuỗi dài  5 có khoảng  38 hàng xóm (1 bước)
Chuỗi dài 10 có khoảng  73 hàng xóm (1 bước)
Chuỗi dài 20 có khoảng 143 hàng xóm (1 bước)
Chuỗi dài 50 có khoảng 353 hàng xóm (1 bước)


Mỗi chuỗi dài `L` có khoảng `7L + 3` hàng xóm. Nếu đáp án là `k` bước, BFS có thể phải đi qua cỡ `(7L)^k` chuỗi — **hàm mũ** theo số bước. Ta viết BFS, đồng thời đếm số chuỗi đã gặp để "đo" độ nặng.

In [8]:
def bfs_search(password: str):
    # Trả về (số bước tối thiểu, số chuỗi đã gặp)
    if is_strong(password):
        return 0, 1
    dist = {password: 0}
    queue = deque([password])
    while queue:
        s = queue.popleft()
        for t in neighbors(s):
            if t in dist:
                continue
            dist[t] = dist[s] + 1
            if is_strong(t):          # BFS: gặp đích lần đầu = ít bước nhất
                return dist[t], len(dist)
            queue.append(t)


def strong_password_bfs(password: str) -> int:
    return bfs_search(password)[0]


for p, expected in [("a", 5), ("aA1", 3), ("1337C0d3", 0), ("aaaaaa", 2), ("......", 3)]:
    print(f"{p!r:12} BFS = {strong_password_bfs(p)}   (mong đợi {expected})")

'a'          BFS = 5   (mong đợi 5)
'aA1'        BFS = 3   (mong đợi 3)
'1337C0d3'   BFS = 0   (mong đợi 0)
'aaaaaa'     BFS = 2   (mong đợi 2)
'......'     BFS = 3   (mong đợi 3)


In [9]:
for k in [9, 10, 11, 12, 13, 14, 15]:
    steps, visited = bfs_search("a" * k)
    print(f"'a'*{k:2}: đáp án {steps} bước, BFS đã gặp {visited:7,} chuỗi")

'a'* 9: đáp án 3 bước, BFS đã gặp     698 chuỗi
'a'*10: đáp án 3 bước, BFS đã gặp   2,124 chuỗi
'a'*11: đáp án 3 bước, BFS đã gặp   3,630 chuỗi
'a'*12: đáp án 4 bước, BFS đã gặp  13,242 chuỗi


'a'*13: đáp án 4 bước, BFS đã gặp  41,236 chuỗi


'a'*14: đáp án 4 bước, BFS đã gặp  71,894 chuỗi


'a'*15: đáp án 5 bước, BFS đã gặp 245,818 chuỗi


Số chuỗi BFS phải xét tăng vọt: từ vài trăm lên hàng trăm nghìn khi chỉ thêm vài ký tự (và mỗi bước thêm vào đáp án nhân số chuỗi lên nhiều lần). Với chuỗi 50 ký tự (đáp án có thể tới hơn 30 bước), BFS là **vô vọng**.

**Cách cũ vướng ở đâu?** BFS không hiểu gì về cấu trúc bài toán: nó thử mù mọi chỗ chèn/xóa/thay. **Mấu chốt cải tiến**: phân tích xem mỗi loại thao tác giải quyết **được gì**, rồi đếm thẳng số bước bằng công thức. BFS vẫn rất quý: ta giữ nó làm **trọng tài** cho mọi chuỗi ngắn.

## 9. Bước đột phá — Ba "món nợ" và mỗi thao tác trả được món nào

Hãy coi một mật khẩu yếu đang **mắc nợ** ba thứ:

1. **Nợ loại ký tự**: `missing` = số loại (thường/hoa/số) còn thiếu, từ `0` tới `3`.
2. **Nợ độ dài**: thiếu `6 - n` ký tự (nếu `n < 6`) hoặc thừa `n - 20` ký tự (nếu `n > 20`).
3. **Nợ đoạn lặp**: mỗi đoạn lặp dài `L >= 3` cần được "bẻ gãy".

Mỗi thao tác trả được món nào?

| Thao tác | Nợ loại ký tự | Nợ độ dài | Nợ đoạn lặp |
|---|---|---|---|
| **Thay** | trả được 1 (thay bằng loại đang thiếu) | không đổi độ dài | rất hiệu quả: 1 lần thay "che" được 3 ký tự |
| **Chèn** | trả được 1 | trả được nếu đang **thiếu** | kém hơn: 1 lần chèn chỉ "che" được 2 ký tự |
| **Xóa** | không bao giờ giúp | trả được nếu đang **thừa** | kém nhất: phải xóa tới khi đoạn còn 2 |

Điểm mấu chốt: một bước có thể trả **hai món nợ cùng lúc** (thay ký tự giữa đoạn lặp bằng một loại đang thiếu). Vì vậy đáp án thường là **`max`** của các món nợ, không phải **tổng**.

### 9.1. Vì sao đoạn lặp dài `L` cần đúng `L // 3` lần thay?

Đoạn lặp dài `L` (ví dụ `"aaaaaaa"`, `L = 7`). Thay ký tự ở các vị trí thứ **3, 6, 9, ...** (đếm từ 1) trong đoạn:

```text
a a a a a a a      (L = 7)
a a X a a X a      → thay vị trí 3 và 6: 2 lần = 7 // 3
```

Mỗi lần thay "đóng một rào chắn" sau 2 ký tự giống nhau, nên các khúc còn lại dài tối đa 2. Số rào cần là `L // 3`.

**Không thể ít hơn**: mỗi lần thay chỉ "phục vụ" được tối đa 3 ký tự (2 ký tự bên trái nó + chính nó), nên `L` ký tự cần ít nhất `L // 3` lần thay. Ở mục 19 ta sẽ kiểm chứng bằng máy một khẳng định mạnh hơn: **bất kỳ** thao tác nào (thay/chèn/xóa) cũng chỉ làm tổng `Σ L // 3` giảm **tối đa 1**.

So sánh với hai thao tác còn lại để thấy thay là "rẻ" nhất khi độ dài đã ổn:

- **Chèn** một ký tự khác vào sau mỗi 2 ký tự: cần `(L - 1) // 2` lần.
- **Xóa** cho tới khi đoạn còn 2: cần `L - 2` lần.

In [10]:
def break_run_by_replace(L, ch="a", new="X"):
    s = list(ch * L)
    for pos in range(2, L, 3):        # vị trí 3, 6, 9, ... (đếm từ 1) = chỉ số 2, 5, 8, ...
        s[pos] = new
    return "".join(s)

for L in [3, 4, 5, 6, 7, 10]:
    t = break_run_by_replace(L)
    print(f"L={L:2}: {'a'*L:12} -> {t:12} thay {t.count('X')} lần (L//3 = {L//3}), còn lặp 3? {has_triple(t)}")


print(" L | thay L//3 | chèn (L-1)//2 | xóa L-2")
for L in range(3, 13):
    print(f"{L:2} | {L//3:9} | {(L-1)//2:13} | {L-2:7}")

L= 3: aaa          -> aaX          thay 1 lần (L//3 = 1), còn lặp 3? False
L= 4: aaaa         -> aaXa         thay 1 lần (L//3 = 1), còn lặp 3? False
L= 5: aaaaa        -> aaXaa        thay 1 lần (L//3 = 1), còn lặp 3? False
L= 6: aaaaaa       -> aaXaaX       thay 2 lần (L//3 = 2), còn lặp 3? False
L= 7: aaaaaaa      -> aaXaaXa      thay 2 lần (L//3 = 2), còn lặp 3? False
L=10: aaaaaaaaaa   -> aaXaaXaaXa   thay 3 lần (L//3 = 3), còn lặp 3? False
 L | thay L//3 | chèn (L-1)//2 | xóa L-2
 3 |         1 |             1 |       1
 4 |         1 |             1 |       2
 5 |         1 |             2 |       3
 6 |         2 |             2 |       4
 7 |         2 |             3 |       5
 8 |         2 |             3 |       6
 9 |         3 |             4 |       7
10 |         3 |             4 |       8
11 |         3 |             5 |       9
12 |         4 |             5 |      10


Thay luôn tốn ít thao tác nhất (bằng hoặc ít hơn chèn, ít hơn hẳn xóa). Vì vậy khi **độ dài đã ổn**, ta đặt:

```text
replace = Σ (L // 3)   lấy tổng trên mọi đoạn lặp có L >= 3
```

Bây giờ chia bài toán theo **độ dài `n`** — có đúng ba trường hợp.

## 10. Trường hợp 1 — `n < 6`: đáp án là `max(missing, 6 - n)`

Bắt buộc phải **chèn ít nhất `6 - n`** ký tự (xóa hay thay đều không làm dài thêm). Cũng phải có ít nhất `missing` bước (mỗi bước chỉ thêm được một loại ký tự). Vậy đáp án **ít nhất** là `max(missing, 6 - n)`.

Điều bất ngờ: **luôn làm được với đúng chừng ấy bước**, không cần quan tâm đoạn lặp! Vì `n <= 5`, đoạn lặp dài nhất chỉ là 5:

- Đoạn dài 3 hoặc 4: chèn **một** ký tự vào giữa (`aa|X|aa`) là xong, và ta có ít nhất 1 lần chèn.
- Đoạn dài 5: khi đó cả chuỗi là `"aaaaa"` (5 ký tự giống hệt) → thiếu ít nhất 2 loại → `missing >= 2`, tức có ít nhất 2 bước: chèn 1 + thay 1: `"aaaaa" → "aaAaaa" → "aaAaa1"`.
- Mọi ký tự chèn/thay đều chọn đúng loại đang thiếu, nên trả nợ loại ký tự cùng lúc.

| password | n | missing | 6 - n | đáp án |
|---|---:|---:|---:|---:|
| `"a"` | 1 | 2 | 5 | **5** |
| `"aA1"` | 3 | 0 | 3 | **3** |
| `"aaaaa"` | 5 | 2 | 1 | **2** |
| `"....."` | 5 | 3 | 1 | **3** |

Chú ý: công thức này **chỉ đúng vì ngưỡng 6 nhỏ**. Ở mục 22 ta sẽ thấy nếu đổi ngưỡng thành 10 thì công thức sai.

In [11]:
def case_short(password: str) -> int:
    return max(count_missing(password), 6 - len(password))

for p in ["a", "aA1", "aaaaa", ".....", "aaa", "Ab1.."]:
    print(f"{p!r:9} công thức = {case_short(p)}   BFS = {strong_password_bfs(p)}")

'a'       công thức = 5   BFS = 5
'aA1'     công thức = 3   BFS = 3
'aaaaa'   công thức = 2   BFS = 2
'.....'   công thức = 3   BFS = 3
'aaa'     công thức = 3   BFS = 3
'Ab1..'   công thức = 1   BFS = 1


Đừng chỉ tin vài ví dụ. Với `n <= 5`, ta có thể kiểm tra **toàn bộ** các chuỗi trên bảng chữ cái thu gọn `"aB1."` (đủ đại diện: một chữ thường, một chữ hoa, một chữ số, một ký tự "không loại"): `4 + 16 + 64 + 256 + 1024 = 1364` chuỗi. Cell này chạy khoảng vài giây.

In [12]:
from itertools import product

checked = 0
for n in range(1, 6):
    for chars in product("aB1.", repeat=n):
        p = "".join(chars)
        assert case_short(p) == strong_password_bfs(p), p
        checked += 1
print("Đã kiểm tra toàn bộ", checked, "chuỗi có n <= 5: công thức khớp BFS")

Đã kiểm tra toàn bộ 1364 chuỗi có n <= 5: công thức khớp BFS


## 11. Trường hợp 2 — `6 <= n <= 20`: đáp án là `max(missing, replace)`

Độ dài đã ổn, nên ta chỉ dùng **thay** (chèn/xóa không rẻ hơn, lại có thể làm lệch độ dài).

- Cần ít nhất `replace = Σ L // 3` bước để bẻ hết đoạn lặp.
- Cần ít nhất `missing` bước để đủ loại.
- **Hai món nợ trả chung được**: mỗi lần thay trong đoạn lặp, ta chọn ký tự thay là **loại đang thiếu**. Nếu `replace >= missing` thì `replace` lần thay vừa bẻ lặp vừa lấp đủ loại. Nếu `missing > replace` thì sau khi bẻ lặp, thay thêm vài ký tự "dư" (không phải đại diện duy nhất của loại nào) bằng loại còn thiếu.

Vậy đáp án = `max(missing, replace)` — **không phải tổng**.

| password | n | đoạn lặp | replace | missing | đáp án |
|---|---:|---|---:|---:|---:|
| `"aaaaaa"` | 6 | `[6]` | 2 | 2 | **2** (`"aaAaa1"`) |
| `"abcdef"` | 6 | không có | 0 | 2 | **2** |
| `"1111111111"` | 10 | `[10]` | 3 | 2 | **3** |
| `"aaaB1bbbbbbb"` | 12 | `[3, 7]` | 3 | 0 | **3** |

In [13]:
def count_replace(password: str) -> int:
    return sum(L // 3 for ch, L in get_runs(password) if L >= 3)

def case_medium(password: str) -> int:
    return max(count_missing(password), count_replace(password))

for p in ["aaaaaa", "abcdef", "1111111111", "aaaB1bbbbbbb", "1337C0d3", "......"]:
    print(f"{p!r:15} replace={count_replace(p)} missing={count_missing(p)} "
          f"-> công thức {case_medium(p)} | BFS {strong_password_bfs(p)}")

'aaaaaa'        replace=2 missing=2 -> công thức 2 | BFS 2
'abcdef'        replace=0 missing=2 -> công thức 2 | BFS 2
'1111111111'    replace=3 missing=2 -> công thức 3 | BFS 3
'aaaB1bbbbbbb'  replace=3 missing=0 -> công thức 3 | BFS 3
'1337C0d3'      replace=0 missing=0 -> công thức 0 | BFS 0
'......'        replace=2 missing=3 -> công thức 3 | BFS 3


## 12. Trường hợp 3 — `n > 20`: phải xóa, nhưng **xóa ở đâu**?

Khi `n > 20`, bắt buộc phải **xóa ít nhất `delete = n - 20`** ký tự. Chèn thì vô ích (chèn 1 lại phải xóa thêm 1). Sau khi xóa đủ, chuỗi dài 20, ta quay về trường hợp 2: tốn thêm `max(missing, replace_mới)`.

```text
đáp án = delete + max(missing, replace sau khi đã xóa)
```

`delete` là cố định, nên câu hỏi duy nhất là: **phân bổ các lần xóa vào đâu để `replace` còn lại nhỏ nhất?** Thử với chuỗi dài 21 (chỉ được xóa 1 ký tự):

```text
"aaaaaBBBBBB1234567890"   đoạn lặp: 5 (aaaaa) và 6 (BBBBBB), missing = 0
```

- Xóa 1 chữ `a` → đoạn 4 và 6 → `replace = 1 + 2 = 3` → tổng `1 + 3 = 4`.
- Xóa 1 chữ `B` → đoạn 5 và 5 → `replace = 1 + 1 = 2` → tổng `1 + 2 = 3`. **Tốt hơn!**

Cell dưới thử xóa **từng vị trí** một và in ra `replace` còn lại.

In [14]:
p = "aaaaaBBBBBB1234567890"
print("n =", len(p), "| đoạn lặp:", [L for ch, L in get_runs(p) if L >= 3], "| replace ban đầu:", count_replace(p))
results = set()
for i in range(len(p)):
    q = p[:i] + p[i + 1:]
    results.add(count_replace(q))
    if i in (0, 5, 11):            # đại diện: một chữ 'a', một chữ 'B', một chữ số
        print(f"xóa vị trí {i:2} (ký tự {p[i]!r}) -> đoạn lặp {[L for ch, L in get_runs(q) if L >= 3]}, replace = {count_replace(q)}")
print("Thử xóa cả 21 vị trí, các giá trị replace có thể:", sorted(results))

n = 21 | đoạn lặp: [5, 6] | replace ban đầu: 3
xóa vị trí  0 (ký tự 'a') -> đoạn lặp [4, 6], replace = 3
xóa vị trí  5 (ký tự 'B') -> đoạn lặp [5, 5], replace = 2
xóa vị trí 11 (ký tự '1') -> đoạn lặp [5, 6], replace = 3
Thử xóa cả 21 vị trí, các giá trị replace có thể: [2, 3]


### 12.1. Quan sát vàng: `L % 3` quyết định "giá" của một lần giảm `replace`

Nhìn lại bảng ở mục 6: bớt 1 ký tự khỏi đoạn dài `L` chỉ làm `L // 3` giảm khi `L % 3 == 0`. Tổng quát hơn, để giảm `replace` đi **1 đơn vị** ở một đoạn, cần xóa:

| `L % 3` | Ví dụ | Xóa bao nhiêu để `L // 3` giảm 1? | Sau đó đoạn có `L % 3` = |
|:---:|---|:---:|:---:|
| `0` | `6 → 5`: `2 → 1` | **1** ký tự | `2` |
| `1` | `7 → 5`: `2 → 1` | **2** ký tự | `2` |
| `2` | `8 → 5`: `2 → 1` | **3** ký tự | `2` |

Để ý cột cuối: sau lần giảm đầu tiên, **mọi đoạn đều có `L % 3 == 2`**, và từ đó mỗi lần giảm tiếp theo đều tốn đúng **3** lần xóa.

Vì vậy mỗi đoạn lặp giống như một "cửa hàng" bán các món "giảm 1 replace" với giá (tính bằng số lần xóa):

```text
đoạn L % 3 == 0:  giá 1, 3, 3, 3, ...
đoạn L % 3 == 1:  giá 2, 3, 3, 3, ...
đoạn L % 3 == 2:  giá 3, 3, 3, 3, ...
```

Ta có ngân sách cố định `delete` lần xóa (bắt buộc phải tiêu) → muốn mua **nhiều món nhất** → mua **món rẻ nhất trước**:

1. Giai đoạn 1: mỗi đoạn `L % 3 == 0` — xóa 1 ký tự, `replace` giảm 1.
2. Giai đoạn 2: mỗi đoạn `L % 3 == 1` — xóa 2 ký tự, `replace` giảm 1.
3. Giai đoạn 3: còn dư bao nhiêu lần xóa thì cứ mỗi 3 lần, `replace` giảm 1.

Trong mỗi cửa hàng giá **không giảm dần** (1 rồi 3, 2 rồi 3, ...), nên "mua rẻ trước" không bao giờ bị mắc kẹt kiểu "phải mua món đắt để mở khóa món rẻ".

In [15]:
def price_list(L):
    # Danh sách số lần xóa cần cho từng lần giảm replace, cho tới khi đoạn còn < 3
    prices = []
    while L >= 3:
        cost = 0
        target = L // 3 - 1
        while L // 3 > target:
            L -= 1
            cost += 1
        prices.append(cost)
    return prices

for L in [3, 4, 5, 6, 7, 8, 9, 10, 11]:
    print(f"L={L:2} (L%3={L%3}): replace ban đầu {L//3}, giá từng lần giảm = {price_list(L)}")

L= 3 (L%3=0): replace ban đầu 1, giá từng lần giảm = [1]
L= 4 (L%3=1): replace ban đầu 1, giá từng lần giảm = [2]
L= 5 (L%3=2): replace ban đầu 1, giá từng lần giảm = [3]
L= 6 (L%3=0): replace ban đầu 2, giá từng lần giảm = [1, 3]
L= 7 (L%3=1): replace ban đầu 2, giá từng lần giảm = [2, 3]
L= 8 (L%3=2): replace ban đầu 2, giá từng lần giảm = [3, 3]
L= 9 (L%3=0): replace ban đầu 3, giá từng lần giảm = [1, 3, 3]
L=10 (L%3=1): replace ban đầu 3, giá từng lần giảm = [2, 3, 3]
L=11 (L%3=2): replace ban đầu 3, giá từng lần giảm = [3, 3, 3]


Bảng giá khớp đúng phân tích: phần tử đầu tiên là `L % 3 + 1`, các phần tử sau đều là `3`.

Còn hai câu hỏi nhỏ:

- **Nếu xóa hết nhu cầu rồi mà vẫn phải xóa tiếp?** (mọi đoạn đã dài `<= 2`) — ta xóa một ký tự "vô hại": ví dụ một ký tự thuộc loại có từ 2 ký tự trở lên hoặc một dấu `.`/`!`, và không xóa ký tự nằm giữa hai ký tự giống nhau (để không nối hai đoạn). Chuỗi dài hơn 20 nên luôn còn chỗ an toàn như vậy; khi đó `replace` không đổi và `missing` không tăng.
- **Có nên xóa để giảm `missing`?** Không — xóa không bao giờ thêm loại ký tự. Nợ loại ký tự vẫn trả bằng các lần thay ở bước sau (hoặc chung với chúng nhờ `max`).

## 13. Cách 1 — Lời giải chính: ba trường hợp + xóa tham lam từng ký tự

Ghép tất cả lại:

```text
missing = số loại còn thiếu
nếu n < 6:        return max(missing, 6 - n)
lengths = độ dài các đoạn lặp có L >= 3
delete = max(0, n - 20)
lặp delete lần:
    chọn đoạn còn L >= 3 có L % 3 NHỎ NHẤT, bớt 1 ký tự của nó
    (nếu không còn đoạn nào: xóa ký tự vô hại, không đổi gì)
replace = Σ L // 3
return delete + max(missing, replace)
```

Chú ý: trường hợp `6 <= n <= 20` nằm sẵn trong code (khi đó `delete = 0`, vòng lặp không chạy).

Vì sao "mỗi lần chọn `L % 3` nhỏ nhất" thực hiện đúng ba giai đoạn ở mục 12.1? Xóa 1 ký tự ở đoạn `% 3 == 1` biến nó thành `% 3 == 0`, nên lần chọn kế tiếp sẽ lại chọn ngay đoạn đó → đúng "giá 2" của giai đoạn 2. Đoạn `% 3 == 2` bị xóa sẽ thành `% 3 == 1` rồi `% 3 == 0` → đúng "giá 3".

In [16]:
def strong_password_greedy(password: str) -> int:
    n = len(password)
    missing = count_missing(password)
    if n < 6:
        return max(missing, 6 - n)

    lengths = [L for ch, L in get_runs(password) if L >= 3]
    delete = max(0, n - 20)
    for _ in range(delete):
        best = -1
        for i, L in enumerate(lengths):
            if L >= 3 and (best == -1 or L % 3 < lengths[best] % 3):
                best = i
        if best == -1:
            break                      # mọi đoạn đã ổn: các lần xóa còn lại là "vô hại"
        lengths[best] -= 1

    replace = sum(L // 3 for L in lengths)
    return delete + max(missing, replace)


for p, expected in [("a", 5), ("aA1", 3), ("1337C0d3", 0)]:
    print(f"{p!r:12} -> {strong_password_greedy(p)} (mong đợi {expected})")
print(strong_password_greedy("aaaaaBBBBBB1234567890"), "<- ví dụ mục 12, mong đợi 3")

'a'          -> 5 (mong đợi 5)
'aA1'        -> 3 (mong đợi 3)
'1337C0d3'   -> 0 (mong đợi 0)
3 <- ví dụ mục 12, mong đợi 3


In [17]:
import random

random.seed(420)
count = 0
for _ in range(300):
    n = random.randint(1, 10)
    p = "".join(random.choice("aaaB1.") for _ in range(n))
    assert strong_password_greedy(p) == strong_password_bfs(p), p
    count += 1
print("Cách 1 khớp BFS trên", count, "chuỗi ngẫu nhiên dài 1..10")

Cách 1 khớp BFS trên 300 chuỗi ngẫu nhiên dài 1..10


## 14. Mô phỏng (trace) từng bước

Chọn một input đủ "khó" để đi qua **cả ba giai đoạn** xóa:

```text
password = "aaaBBBB11111ccccccxY2zW3vQ!"     n = 27  →  delete = 7
đoạn lặp:  aaa (3, %3=0) | BBBB (4, %3=1) | 11111 (5, %3=2) | cccccc (6, %3=0)
replace ban đầu = 1 + 1 + 1 + 2 = 5,  missing = 0
```

Bảng dự kiến (mỗi dòng là một lần xóa, chọn đoạn có `L % 3` nhỏ nhất, hòa thì lấy đoạn đứng trước):

| Lần xóa | lengths trước | Chọn | Giải thích | replace sau |
|---:|---|---|---|---:|
| 1 | `[3, 4, 5, 6]` | đoạn 0: 3 → 2 | `%3 = 0`, giá 1 | 4 |
| 2 | `[2, 4, 5, 6]` | đoạn 3: 6 → 5 | `%3 = 0`, giá 1 | 3 |
| 3 | `[2, 4, 5, 5]` | đoạn 1: 4 → 3 | `%3 = 1`, nửa đầu của giá 2 | 3 |
| 4 | `[2, 3, 5, 5]` | đoạn 1: 3 → 2 | giờ `%3 = 0` → trả đủ giá 2 | 2 |
| 5 | `[2, 2, 5, 5]` | đoạn 2: 5 → 4 | chỉ còn `%3 = 2`: giá 3 | 2 |
| 6 | `[2, 2, 4, 5]` | đoạn 2: 4 → 3 | | 2 |
| 7 | `[2, 2, 3, 5]` | đoạn 2: 3 → 2 | trả đủ giá 3 | 1 |

Kết quả: `delete + max(missing, replace) = 7 + max(0, 1) = 8`.

In [18]:
def trace_greedy(password: str):
    n = len(password)
    missing = count_missing(password)
    print(f"password={password!r}  n={n}  missing={missing}")
    if n < 6:
        print(f"n < 6 -> max(missing, 6-n) = max({missing}, {6-n}) = {max(missing, 6-n)}")
        return max(missing, 6 - n)
    lengths = [L for ch, L in get_runs(password) if L >= 3]
    delete = max(0, n - 20)
    print(f"đoạn lặp >= 3: {lengths}  replace ban đầu = {sum(L//3 for L in lengths)}  delete = {delete}")
    for step in range(1, delete + 1):
        best = -1
        for i, L in enumerate(lengths):
            if L >= 3 and (best == -1 or L % 3 < lengths[best] % 3):
                best = i
        if best == -1:
            print(f"  lần {step}: không còn đoạn >= 3 -> xóa ký tự vô hại")
            continue
        before = lengths[:]
        lengths[best] -= 1
        print(f"  lần {step}: {before} chọn đoạn {best} ({before[best]}, %3={before[best]%3})"
              f" -> {lengths}, replace = {sum(L//3 for L in lengths if L >= 3)}")
    replace = sum(L // 3 for L in lengths if L >= 3)
    ans = delete + max(missing, replace)
    print(f"đáp án = {delete} + max({missing}, {replace}) = {ans}")
    return ans

trace_greedy("aaaBBBB11111ccccccxY2zW3vQ!");

password='aaaBBBB11111ccccccxY2zW3vQ!'  n=27  missing=0
đoạn lặp >= 3: [3, 4, 5, 6]  replace ban đầu = 5  delete = 7
  lần 1: [3, 4, 5, 6] chọn đoạn 0 (3, %3=0) -> [2, 4, 5, 6], replace = 4
  lần 2: [2, 4, 5, 6] chọn đoạn 3 (6, %3=0) -> [2, 4, 5, 5], replace = 3
  lần 3: [2, 4, 5, 5] chọn đoạn 1 (4, %3=1) -> [2, 3, 5, 5], replace = 3
  lần 4: [2, 3, 5, 5] chọn đoạn 1 (3, %3=0) -> [2, 2, 5, 5], replace = 2
  lần 5: [2, 2, 5, 5] chọn đoạn 2 (5, %3=2) -> [2, 2, 4, 5], replace = 2
  lần 6: [2, 2, 4, 5] chọn đoạn 2 (4, %3=1) -> [2, 2, 3, 5], replace = 2
  lần 7: [2, 2, 3, 5] chọn đoạn 2 (3, %3=0) -> [2, 2, 2, 5], replace = 1
đáp án = 7 + max(0, 1) = 8


Output khớp từng dòng với bảng dự kiến. Hãy để ý lần xóa 3–4: đoạn `BBBB` (dài 4, `%3 = 1`) bị xóa **hai lần liên tiếp** — đúng "giá 2". Lần 5–7: đoạn `11111` (dài 5, `%3 = 2`) bị xóa **ba lần liên tiếp** — đúng "giá 3".

Thêm vài trace ngắn cho hai trường hợp còn lại và cho một chuỗi mà xóa xong vẫn còn dư lần xóa:

In [19]:
trace_greedy("aaaaa")
print()
trace_greedy("aaaB1bbbbbbb")
print()
trace_greedy("aaaaaaaaaaaaaaaaaaaaaaaaa")      # 25 chữ 'a'
print()
trace_greedy("abcdefghijABCDEFGHIJ1234");      # dài 24, không có đoạn lặp

password='aaaaa'  n=5  missing=2
n < 6 -> max(missing, 6-n) = max(2, 1) = 2

password='aaaB1bbbbbbb'  n=12  missing=0
đoạn lặp >= 3: [3, 7]  replace ban đầu = 3  delete = 0
đáp án = 0 + max(0, 3) = 3

password='aaaaaaaaaaaaaaaaaaaaaaaaa'  n=25  missing=2
đoạn lặp >= 3: [25]  replace ban đầu = 8  delete = 5
  lần 1: [25] chọn đoạn 0 (25, %3=1) -> [24], replace = 8
  lần 2: [24] chọn đoạn 0 (24, %3=0) -> [23], replace = 7
  lần 3: [23] chọn đoạn 0 (23, %3=2) -> [22], replace = 7
  lần 4: [22] chọn đoạn 0 (22, %3=1) -> [21], replace = 7
  lần 5: [21] chọn đoạn 0 (21, %3=0) -> [20], replace = 6
đáp án = 5 + max(2, 6) = 11

password='abcdefghijABCDEFGHIJ1234'  n=24  missing=0
đoạn lặp >= 3: []  replace ban đầu = 0  delete = 4
  lần 1: không còn đoạn >= 3 -> xóa ký tự vô hại
  lần 2: không còn đoạn >= 3 -> xóa ký tự vô hại
  lần 3: không còn đoạn >= 3 -> xóa ký tự vô hại
  lần 4: không còn đoạn >= 3 -> xóa ký tự vô hại
đáp án = 4 + max(0, 0) = 4


Đọc kỹ trace thứ ba (`'a' * 25`): một đoạn dài 25 (`%3 = 1`), `replace = 8`. Hai lần xóa đầu (giá 2) → còn 23 → `replace = 7`; ba lần xóa còn lại vừa đúng một "giá 3" → còn 20 → `replace = 6`. Đáp án `5 + max(2, 6) = 11`: 5 lần xóa + 6 lần thay (trong đó 2 lần thay chọn `A` và `1` để trả luôn nợ loại ký tự).

Trace cuối: không có đoạn lặp nào, 4 lần xóa đều "vô hại", đáp án chỉ là `delete = 4`.

## 15. Cách 2 — Mảng đếm theo mod 3 (tính thẳng, không mô phỏng)

Cách 1 mô phỏng từng lần xóa, mỗi lần quét lại danh sách đoạn. Vì `n <= 50` thì vậy là đủ nhanh, nhưng ta có thể làm **gọn và đẹp hơn**: chỉ cần **đếm** có bao nhiêu đoạn mỗi loại `L % 3`:

```text
cnt[0] = số đoạn lặp (L >= 3) có L % 3 == 0
cnt[1] = số đoạn lặp (L >= 3) có L % 3 == 1
rem    = delete (số lần xóa còn lại để phân bổ)
```

Rồi đi qua ba giai đoạn bằng công thức:

```text
Giai đoạn 1:  use = min(rem, cnt[0])        replace -= use        rem -= use
Giai đoạn 2:  use = min(rem, 2 * cnt[1])    replace -= use // 2   rem -= use
Giai đoạn 3:                                 replace -= rem // 3   (không xuống dưới 0)
```

Giải thích từng dòng bằng ví dụ ở mục 14 (`cnt = [2, 1, 1]`, `delete = 7`, `replace = 5`):

- Giai đoạn 1: có 2 đoạn `%3 == 0`, đủ xóa → `use = min(7, 2) = 2`, `replace = 3`, `rem = 5`.
- Giai đoạn 2: 1 đoạn `%3 == 1`, cần 2 lần xóa → `use = min(5, 2) = 2`, `replace -= 2 // 2 = 1` → `2`, `rem = 3`.
- Giai đoạn 3: mọi đoạn giờ đều `%3 == 2`, cứ 3 lần xóa giảm 1 → `replace -= 3 // 3 = 1` → `1`.

Đáp án `7 + max(0, 1) = 8` — giống hệt cách 1.

Tại sao giai đoạn 2 dùng `use // 2`? Mỗi đoạn `%3 == 1` cần **đủ 2** lần xóa mới giảm được 1. Nếu `use` lẻ (ví dụ `rem = 3`, `cnt[1] = 3`), ta xóa 2 ký tự ở đoạn thứ nhất (giảm 1) và 1 ký tự ở đoạn thứ hai (mới "trả trước" một nửa giá, chưa giảm được gì) → giảm `3 // 2 = 1`. Lần xóa lẻ cuối cùng vẫn phải tốn (xóa là bắt buộc), chỉ là không làm `replace` giảm.

Tại sao giai đoạn 3 không cần biết đoạn nào? Vì sau hai giai đoạn đầu, **mọi** đoạn còn `L >= 3` đều có `L % 3 == 2`, giá như nhau (3 lần xóa / 1 đơn vị), xóa ở đâu cũng vậy.

In [20]:
def strong_password_mod3(password: str) -> int:
    n = len(password)
    missing = count_missing(password)
    if n < 6:
        return max(missing, 6 - n)

    replace = 0
    cnt = [0, 0, 0]
    for ch, L in get_runs(password):
        if L >= 3:
            replace += L // 3
            cnt[L % 3] += 1
    if n <= 20:
        return max(missing, replace)

    delete = n - 20
    rem = delete
    use = min(rem, cnt[0]);      replace -= use;      rem -= use     # giai đoạn 1
    use = min(rem, 2 * cnt[1]);  replace -= use // 2; rem -= use     # giai đoạn 2
    replace = max(0, replace - rem // 3)                             # giai đoạn 3
    return delete + max(missing, replace)


for p in ["a", "aA1", "1337C0d3", "aaaaaBBBBBB1234567890", "aaaBBBB11111ccccccxY2zW3vQ!", "a" * 25]:
    print(f"{p!r:30} mod3 = {strong_password_mod3(p):2} | greedy = {strong_password_greedy(p):2}")

'a'                            mod3 =  5 | greedy =  5
'aA1'                          mod3 =  3 | greedy =  3
'1337C0d3'                     mod3 =  0 | greedy =  0
'aaaaaBBBBBB1234567890'        mod3 =  3 | greedy =  3
'aaaBBBB11111ccccccxY2zW3vQ!'  mod3 =  8 | greedy =  8
'aaaaaaaaaaaaaaaaaaaaaaaaa'    mod3 = 11 | greedy = 11


Trace của cách 2 in ra **mảng đếm** và trạng thái sau từng giai đoạn. Chọn thêm một input có `rem` lẻ ở giai đoạn 2 để thấy `use // 2`:

```text
"aaaaBBBB1111cdefghijklm"   n = 23 → delete = 3
ba đoạn dài 4 (%3 = 1) → cnt = [0, 3, 0], replace = 3
```

Giai đoạn 2: `use = min(3, 6) = 3`, `replace -= 3 // 2 = 1` → từ `3` còn `2`.

In [21]:
def trace_mod3(password: str):
    n = len(password)
    missing = count_missing(password)
    replace, cnt = 0, [0, 0, 0]
    for ch, L in get_runs(password):
        if L >= 3:
            replace += L // 3
            cnt[L % 3] += 1
    print(f"{password!r}: n={n} missing={missing} replace={replace} cnt={cnt}")
    if n < 6 or n <= 20:
        print("  (không cần xóa)")
        return
    rem = delete = n - 20
    use = min(rem, cnt[0]); replace -= use; rem -= use
    print(f"  GĐ1: dùng {use} lần xóa -> replace={replace}, rem={rem}")
    use = min(rem, 2 * cnt[1]); replace -= use // 2; rem -= use
    print(f"  GĐ2: dùng {use} lần xóa -> replace={replace}, rem={rem}")
    replace = max(0, replace - rem // 3)
    print(f"  GĐ3: {rem} lần xóa còn lại giảm {rem // 3} -> replace={replace}")
    print(f"  đáp án = {delete} + max({missing}, {replace}) = {delete + max(missing, replace)}")

trace_mod3("aaaBBBB11111ccccccxY2zW3vQ!")
trace_mod3("aaaaBBBB1111cdefghijklm")

'aaaBBBB11111ccccccxY2zW3vQ!': n=27 missing=0 replace=5 cnt=[2, 1, 1]
  GĐ1: dùng 2 lần xóa -> replace=3, rem=5
  GĐ2: dùng 2 lần xóa -> replace=2, rem=3
  GĐ3: 3 lần xóa còn lại giảm 1 -> replace=1
  đáp án = 7 + max(0, 1) = 8
'aaaaBBBB1111cdefghijklm': n=23 missing=0 replace=3 cnt=[0, 3, 0]
  GĐ1: dùng 0 lần xóa -> replace=3, rem=3
  GĐ2: dùng 3 lần xóa -> replace=2, rem=0
  GĐ3: 0 lần xóa còn lại giảm 0 -> replace=2
  đáp án = 3 + max(0, 2) = 5


## 16. Code nộp LeetCode

Gom cách 2 vào một method duy nhất, tách đoạn lặp bằng hai con trỏ ngay trong vòng lặp (không tạo danh sách `runs`), nên bộ nhớ phụ chỉ là vài biến và mảng `cnt` 3 phần tử.

In [22]:
class Solution:
    def strongPasswordChecker(self, password: str) -> int:
        n = len(password)
        missing = 3 - (any(c.islower() for c in password)
                       + any(c.isupper() for c in password)
                       + any(c.isdigit() for c in password))
        if n < 6:
            return max(missing, 6 - n)

        replace = 0
        cnt = [0, 0, 0]            # cnt[r] = số đoạn lặp (dài >= 3) có độ dài % 3 == r
        i = 0
        while i < n:
            j = i
            while j < n and password[j] == password[i]:
                j += 1
            L = j - i
            if L >= 3:
                replace += L // 3
                cnt[L % 3] += 1
            i = j

        if n <= 20:
            return max(missing, replace)

        delete = n - 20
        rem = delete
        use = min(rem, cnt[0])          # mỗi đoạn %3==0: xóa 1 -> giảm 1 lần thay
        replace -= use
        rem -= use
        use = min(rem, 2 * cnt[1])      # mỗi đoạn %3==1: xóa 2 -> giảm 1 lần thay
        replace -= use // 2
        rem -= use
        replace = max(0, replace - rem // 3)   # còn lại: xóa 3 -> giảm 1 lần thay
        return delete + max(missing, replace)

In [23]:
sol = Solution()
tests = [
    ("a", 5), ("aA1", 3), ("1337C0d3", 0),            # ví dụ đề
    ("aaaaa", 2), (".....", 3), ("aaaaaa", 2),         # n < 6 và n = 6
    ("abcdef", 2), ("1111111111", 3), ("aaaB1bbbbbbb", 3),
    ("aaaaaBBBBBB1234567890", 3),                     # n = 21: xóa ở đoạn %3==0
    ("aaaaaBBB1c2d3e4f5g6h7", 2),                     # xóa ở đoạn ngắn hơn mới đúng
    ("aaaBBBB11111ccccccxY2zW3vQ!", 8),               # đủ ba giai đoạn
    ("a" * 25, 11), ("." * 50, 36),                    # cực trị
    ("ABABABABABABABABABABABABA", 7),                 # dài 25, không lặp, thiếu 2 loại
    ("abcdefghijABCDEFGHIJ1234", 4),
]
for p, expected in tests:
    got = sol.strongPasswordChecker(p)
    assert got == expected, (p, got, expected)
print("Tất cả", len(tests), "test PASS")

Tất cả 16 test PASS


## 17. Trọng tài cho chuỗi dài: quy hoạch động (DP)

BFS chỉ kiểm chứng được chuỗi ngắn (`n <= 10`), mà phần khó nhất — **`n > 20`** — lại nằm ngoài tầm với của BFS. Để kiểm chứng phần đó một cách độc lập, ta viết một trọng tài khác: **quy hoạch động** (Dynamic Programming — DP), giống tinh thần bài Edit Distance.

Ý tưởng: đọc `password` từ trái sang phải, từng ký tự gốc. Tại mỗi ký tự, ta có 3 lựa chọn: **giữ** (0 bước), **xóa** (1 bước), **thay** bằng ký tự mới (1 bước); ngoài ra lúc nào cũng có thể **chèn** ký tự mới (1 bước). Ta không cần nhớ cả chuỗi kết quả, chỉ cần nhớ đủ thông tin để biết tương lai còn hợp lệ không — gọi là **trạng thái**:

| Thành phần | Ý nghĩa | Giá trị |
|---|---|---|
| `last` | ký tự cuối của chuỗi kết quả (hoặc `"*"` nếu đó là ký tự mới) | ký tự gốc / `"*"` / `None` |
| `run` | ký tự cuối đang lặp mấy lần liên tiếp | `1` hoặc `2` (không bao giờ 3) |
| `mask` | đã có những loại ký tự nào | `0..7` (bitmask, xem dưới) |
| `length` | độ dài chuỗi kết quả | `0..20` |

Ký tự mới luôn được coi là `"*"` — **khác mọi hàng xóm** (đúng lập luận "3 ký tự `a`, `A`, `1` là đủ" ở mục 8), nên ta chỉ cần chọn **loại** của nó. `"*"` không bao giờ trùng ký tự gốc vì input chỉ gồm chữ, số, `.` và `!`.

`dp` là một `dict`: trạng thái → số bước ít nhất để đạt trạng thái đó.

### 17.1. Kiến thức nền 6 — Bitmask: nhớ "đã có loại nào" bằng một số nguyên

Gán mỗi loại một **bit**: chữ thường = `1` (nhị phân `001`), chữ hoa = `2` (`010`), chữ số = `4` (`100`). Phép `|` (OR theo bit) "bật" bit lên:

```text
mask = 0            000   chưa có gì
mask |= 1   -> 1    001   có chữ thường
mask |= 4   -> 5    101   có chữ thường và chữ số
mask |= 1   -> 5    101   bật lại bit đã bật: không đổi
mask |= 2   -> 7    111   đủ cả ba loại
```

Vậy "đủ ba loại" chỉ là `mask == 7`. Một số nguyên nhỏ thay cho ba biến `True/False` — rất tiện làm một phần của trạng thái DP.

In [24]:
def char_bit(c: str) -> int:
    if c.islower(): return 1
    if c.isupper(): return 2
    if c.isdigit(): return 4
    return 0                       # '.' hoặc '!'

mask = 0
for c in "a1a.B":
    mask |= char_bit(c)
    print(f"thêm {c!r}: mask = {mask} = {mask:03b} (nhị phân)  đủ loại? {mask == 7}")

thêm 'a': mask = 1 = 001 (nhị phân)  đủ loại? False
thêm '1': mask = 5 = 101 (nhị phân)  đủ loại? False
thêm 'a': mask = 5 = 101 (nhị phân)  đủ loại? False
thêm '.': mask = 5 = 101 (nhị phân)  đủ loại? False
thêm 'B': mask = 7 = 111 (nhị phân)  đủ loại? True


In [25]:
def strong_password_dp(password: str, lo: int = 6, hi: int = 20) -> int:
    INF = float("inf")

    def push(table, state, cost):
        if state is not None and cost < table.get(state, INF):
            table[state] = cost

    def add_original(state, ch):             # nối một ký tự GỐC vào cuối
        last, run, mask, length = state
        if length == hi:
            return None
        if ch == last:
            if run == 2:
                return None                  # sẽ thành 3 ký tự lặp: cấm
            return (last, run + 1, mask | char_bit(ch), length + 1)
        return (ch, 1, mask | char_bit(ch), length + 1)

    def add_new(state, bit):                 # nối một ký tự MỚI loại `bit`
        last, run, mask, length = state
        if length == hi:
            return None
        return ("*", 1, mask | bit, length + 1)

    def close_with_inserts(layer):           # chèn thêm 0, 1, 2, ... ký tự mới
        for length in range(hi + 1):
            for state in [s for s in layer if s[3] == length]:
                for bit in (1, 2, 4):
                    push(layer, add_new(state, bit), layer[state] + 1)
        return layer

    layer = close_with_inserts({(None, 0, 0, 0): 0})
    for ch in password:
        nxt = {}
        for state, cost in layer.items():
            push(nxt, state, cost + 1)                      # xóa ch
            push(nxt, add_original(state, ch), cost)        # giữ ch
            for bit in (1, 2, 4):
                push(nxt, add_new(state, bit), cost + 1)    # thay ch bằng ký tự mới
        layer = close_with_inserts(nxt)
    return min(cost for (last, run, mask, length), cost in layer.items()
               if mask == 7 and length >= lo)


for p, expected in tests[:6]:
    print(f"{p!r:12} DP = {strong_password_dp(p)} (mong đợi {expected})")

'a'          DP = 5 (mong đợi 5)
'aA1'        DP = 3 (mong đợi 3)


'1337C0d3'   DP = 0 (mong đợi 0)
'aaaaa'      DP = 2 (mong đợi 2)
'.....'      DP = 3 (mong đợi 3)


'aaaaaa'     DP = 2 (mong đợi 2)


Hàm `close_with_inserts` xử lý theo `length` tăng dần: chèn một ký tự làm `length` tăng 1, nên khi tới lớp `length + 1` thì mọi cách chèn từ lớp `length` đã được tính xong. Số trạng thái mỗi lớp chỉ cỡ vài nghìn (`last` × `run` × `mask` × `length`), nên DP chạy khoảng vài chục mili-giây cho một chuỗi 50 ký tự — chậm hơn công thức nhiều, nhưng đủ để làm trọng tài cho hàng chục chuỗi dài.

## 18. Test đối chiếu chéo tất cả các cách

Hai tầng kiểm tra, đều với `random.seed` cố định để kết quả lặp lại được:

1. **Chuỗi ngắn** (`1..10` ký tự): BFS (đúng tuyệt đối) vs DP vs cách 1 vs cách 2 vs `Solution`.
2. **Chuỗi dài** (`21..50` ký tự, BFS bó tay): DP vs cách 1 vs cách 2 vs `Solution`. Sinh chuỗi có nhiều đoạn lặp dài để các giai đoạn xóa thực sự xảy ra.

Cell thứ hai mất khoảng vài giây vì DP.

In [26]:
random.seed(2024)
for _ in range(200):
    n = random.randint(1, 10)
    p = "".join(random.choice("aaAB11.!") for _ in range(n))
    expected = strong_password_bfs(p)
    got = [strong_password_dp(p), strong_password_greedy(p), strong_password_mod3(p), sol.strongPasswordChecker(p)]
    assert all(g == expected for g in got), (p, expected, got)
print("200 chuỗi ngắn: BFS = DP = cách 1 = cách 2 = Solution")


def random_runny_password(n):
    # ghép các khúc lặp có độ dài ngẫu nhiên để tạo nhiều đoạn L >= 3
    alphabet = random.choice(["aB1.", "aaab", "A1!", "a", "aA"])
    s = ""
    while len(s) < n:
        s += random.choice(alphabet) * random.randint(1, 9)
    return s[:n]

random.seed(7)
long_cases = 0
for _ in range(70):
    p = random_runny_password(random.randint(21, 50))
    expected = strong_password_dp(p)
    got = [strong_password_greedy(p), strong_password_mod3(p), sol.strongPasswordChecker(p)]
    assert all(g == expected for g in got), (p, expected, got)
    long_cases += 1
print(long_cases, "chuỗi dài 21..50: DP = cách 1 = cách 2 = Solution")

200 chuỗi ngắn: BFS = DP = cách 1 = cách 2 = Solution


70 chuỗi dài 21..50: DP = cách 1 = cách 2 = Solution


## 19. Vì sao lời giải đúng?

Ta cần hai chiều: (a) **làm được** với số bước công thức đưa ra (cận trên), (b) **không thể ít hơn** (cận dưới).

**Trường hợp `n < 6`** — (b) phải chèn `6 - n` lần và mỗi bước thêm tối đa 1 loại → `>= max(missing, 6 - n)`. (a) đã chỉ ra ở mục 10: đoạn lặp dài tối đa 5 nên luôn bẻ được bằng chính các bước đó.

**Trường hợp `6 <= n <= 20`** — (a) thay các vị trí 3, 6, 9... của từng đoạn, ưu tiên ký tự thay là loại còn thiếu. (b) Mỗi bước trả tối đa 1 loại → `>= missing`. Và **mỗi thao tác chỉ làm `Σ L // 3` giảm tối đa 1** (khẳng định quan trọng nhất) → `>= replace`. Cell dưới kiểm chứng khẳng định này bằng máy: với mọi đoạn `L <= 15`, thử **mọi** cách xóa/chèn/thay một vị trí, đo `Σ L // 3` sau thao tác.

**Trường hợp `n > 20`** — bắt buộc tốn `delete = n - 20` lần xóa; chèn thì vô ích (tốn thêm 1 lần xóa). Sau khi xóa, phần còn lại đúng là trường hợp 2 với chuỗi dài 20. Ta chỉ còn phải chọn **cách phân bổ** các lần xóa sao cho `replace` nhỏ nhất. Mỗi đoạn là một "cửa hàng" với bảng giá không giảm (`L%3 + 1`, rồi `3, 3, ...`); với ngân sách cố định, mua món rẻ nhất trước luôn mua được nhiều món nhất — đó chính là ba giai đoạn. (Lập luận đầy đủ hơn cần xét cả việc một lần thay có thể "dùng chung" cho nhiều điều kiện; trọng tài DP ở mục 18 xác nhận công thức trên hàng chục chuỗi dài ngẫu nhiên.)

**Bất biến (invariant) của cách 1, nói bằng lời**: sau mỗi lần xóa, `Σ L // 3` hiện tại là **nhỏ nhất có thể** với số lần xóa đã dùng.

In [27]:
def replace_cost(s):
    return sum(L // 3 for ch, L in get_runs(s) if L >= 3)

worst_drop = 0
for L in range(1, 16):
    s = "a" * L + "Z"                 # thêm 'Z' ở cuối để có hàng xóm khác
    before = replace_cost(s)
    for t in neighbors(s) + [s[:i] + "Z" + s[i+1:] for i in range(len(s))]:
        worst_drop = max(worst_drop, before - replace_cost(t))
print("Một thao tác làm Σ L//3 giảm nhiều nhất:", worst_drop)

Một thao tác làm Σ L//3 giảm nhiều nhất: 1


## 20. Độ phức tạp và so sánh các cách

Gọi `n = len(password)` (`n <= 50`), `k` = số đoạn lặp dài `>= 3` (`k <= n / 3`), `d = max(0, n - 20)` (`d <= 30`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Vai trò |
|---|---|---|---|---|
| 0. BFS trên chuỗi | thử mọi chuỗi theo số bước | hàm mũ theo đáp án, cỡ `(7n)^ans` | hàm mũ | trọng tài cho `n <= 10` |
| DP trọng tài | duyệt ký tự gốc, trạng thái `(last, run, mask, length)` | `O(n · S)` với `S` = vài nghìn trạng thái | `O(S)` | trọng tài cho `n > 20` |
| 1. Ba trường hợp + xóa tham lam | mỗi lần xóa chọn đoạn `L % 3` nhỏ nhất | `O(n + d · k)` | `O(k)` | dễ hiểu, dễ trace |
| 2. Mảng đếm mod 3 | `cnt[0]`, `cnt[1]` + 3 công thức | `O(n)` | `O(1)` (bản `Solution`) | **bản nộp** |

Vì `n <= 50`, cả cách 1 và cách 2 đều chạy tức thì; cách 2 được ưa chuộng vì code ngắn và không phải mô phỏng. Cách 1 lại là bước trung gian rất quan trọng để **hiểu vì sao** công thức của cách 2 đúng.

Ghi chú về `O(n)` của cách 2: tách đoạn lặp bằng hai con trỏ đi qua mỗi ký tự một lần; ba lần `any(...)` mỗi lần `O(n)`; ba giai đoạn chỉ là vài phép tính `O(1)`.

## 21. Lỗi thường gặp (code sai → vì sao → sửa)

**Lỗi 1 — Cộng thay vì lấy `max`** trong trường hợp `6..20`:

```python
return missing + replace          # SAI
return max(missing, replace)      # ĐÚNG
```

Vì sao sai: một lần thay giữa đoạn lặp có thể chọn đúng loại đang thiếu, trả **hai món nợ cùng lúc**. `"aaaaaa"`: sai ra `2 + 2 = 4`, đúng là `2` (`"aaAaa1"`).

**Lỗi 2 — Cộng thay vì lấy `max`** trong trường hợp `n < 6`: `6 - n + missing`. `"a"`: sai ra `5 + 2 = 7`, đúng là `5` (các ký tự chèn thêm chính là loại đang thiếu).

**Lỗi 3 — Xóa "đại" ở đoạn đầu tiên / đoạn dài nhất** khi `n > 20`, không để ý `L % 3`. Mục 12 đã cho phản ví dụ.

**Lỗi 4 — Coi đoạn `L % 3 == 1` như đoạn `L % 3 == 0`** ở giai đoạn 2 (`replace -= min(rem, cnt[1])`): đoạn dài 4 xóa 1 ký tự vẫn còn 3, vẫn cần 1 lần thay.

**Lỗi 5 — Quên cộng `delete`** vào đáp án khi `n > 20` (chỉ trả `max(missing, replace)`).

In [28]:
def wrong_sum(password):                                  # lỗi 1 + lỗi 2
    n, missing, replace = len(password), count_missing(password), count_replace(password)
    if n < 6:
        return missing + (6 - n)
    if n <= 20:
        return missing + replace
    return strong_password_mod3(password)

def wrong_delete_first_run(password):                     # lỗi 3a: xóa ở đoạn đầu tiên
    n, missing = len(password), count_missing(password)
    if n <= 20:
        return strong_password_mod3(password)
    lengths = [L for ch, L in get_runs(password) if L >= 3]
    for _ in range(n - 20):
        for i, L in enumerate(lengths):
            if L >= 3:
                lengths[i] -= 1
                break
    return (n - 20) + max(missing, sum(L // 3 for L in lengths))

def wrong_delete_longest(password):                       # lỗi 3b: xóa ở đoạn dài nhất
    n, missing = len(password), count_missing(password)
    if n <= 20:
        return strong_password_mod3(password)
    lengths = [L for ch, L in get_runs(password) if L >= 3]
    for _ in range(n - 20):
        if lengths and max(lengths) >= 3:
            lengths[lengths.index(max(lengths))] -= 1
    return (n - 20) + max(missing, sum(L // 3 for L in lengths))

for f, p in [(wrong_sum, "aaaaaa"), (wrong_sum, "a"),
             (wrong_delete_first_run, "aaaaaBBBBBB1234567890"),
             (wrong_delete_longest, "aaaaaBBB1c2d3e4f5g6h7")]:
    print(f"{f.__name__:24} {p!r:25} sai = {f(p)} | đúng = {strong_password_mod3(p)}")


def wrong_phase2(password):                               # lỗi 4
    n, missing = len(password), count_missing(password)
    if n <= 20:
        return strong_password_mod3(password)
    replace, cnt = 0, [0, 0, 0]
    for ch, L in get_runs(password):
        if L >= 3:
            replace += L // 3
            cnt[L % 3] += 1
    rem = n - 20
    use = min(rem, cnt[0]); replace -= use; rem -= use
    use = min(rem, cnt[1]); replace -= use; rem -= use      # SAI: mỗi đoạn %3==1 cần 2 lần xóa
    replace = max(0, replace - rem // 3)
    return (n - 20) + max(missing, replace)

def wrong_forget_delete(password):                        # lỗi 5
    return strong_password_mod3(password) - max(0, len(password) - 20)

p = "aaaaB1c2d3e4f5g6h7i8j"
print(f"wrong_phase2        {p!r}: sai = {wrong_phase2(p)} | đúng = {strong_password_mod3(p)}")
p = "a" * 25
print(f"wrong_forget_delete {p!r}: sai = {wrong_forget_delete(p)} | đúng = {strong_password_mod3(p)}")

wrong_sum                'aaaaaa'                  sai = 4 | đúng = 2
wrong_sum                'a'                       sai = 7 | đúng = 5
wrong_delete_first_run   'aaaaaBBBBBB1234567890'   sai = 4 | đúng = 3
wrong_delete_longest     'aaaaaBBB1c2d3e4f5g6h7'   sai = 3 | đúng = 2
wrong_phase2        'aaaaB1c2d3e4f5g6h7i8j': sai = 1 | đúng = 2
wrong_forget_delete 'aaaaaaaaaaaaaaaaaaaaaaaaa': sai = 6 | đúng = 11


## 22. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu nhận dạng bài "phân tích trường hợp + tham lam"**:

- Đề hỏi "ít nhất bao nhiêu bước", có **vài điều kiện độc lập** cần thỏa cùng lúc, và một bước có thể giúp nhiều điều kiện → nghĩ tới **`max` của các cận dưới**, rồi chứng minh cận đó đạt được.
- Có ràng buộc dạng **"không quá `k` phần tử giống nhau liên tiếp"** → tách thành **đoạn lặp** và nhìn độ dài theo **mod `(k+1)`** (ở đây `k = 2`, mod 3).
- Ngân sách thao tác **bắt buộc phải tiêu** (ở đây `n - 20` lần xóa) → tận dụng nó để giảm chi phí khác, **mua món rẻ nhất trước**.
- Luôn viết một **trọng tài vét cạn** (BFS/DP) trên input nhỏ trước khi tin công thức.

**Bài liên quan trên LeetCode**:

- **2299. Strong Password Checker II** (Easy): chỉ hỏi mạnh hay không (điều kiện khác một chút) — luyện phần `is_strong`.
- **1957. Delete Characters to Make Fancy String** (Easy): xóa ít nhất để không có 3 ký tự giống nhau liên tiếp — đúng "món nợ đoạn lặp" nhưng chỉ được xóa.
- **1578. Minimum Time to Make Rope Colorful** (Medium): xử lý theo từng đoạn lặp, mỗi đoạn giữ lại một phần tử.
- **72. Edit Distance** (Medium): cùng ba thao tác chèn/xóa/thay, lời giải DP — họ hàng của trọng tài DP ở mục 17.

**Biến thể: đổi ngưỡng độ dài**. Nếu đề đổi thành "ít nhất 10 ký tự", công thức `max(missing, lo - n)` của trường hợp ngắn **không còn đúng**, vì đoạn lặp có thể dài tới 9 mà chỉ có 1 lần chèn. Trọng tài DP (đã viết sẵn tham số `lo`, `hi`) chỉ ra ngay:

In [29]:
p = "aaaaaaaaa"                  # 9 chữ 'a'
naive = max(count_missing(p), 10 - len(p))
print(f"lo=10: công thức ngây thơ max(missing, lo-n) = {naive}, DP = {strong_password_dp(p, lo=10, hi=20)}")
print(f"lo=6 : công thức gốc (ở đây n=9 thuộc trường hợp 2) = {strong_password_mod3(p)}, DP = {strong_password_dp(p)}")

lo=10: công thức ngây thơ max(missing, lo-n) = 2, DP = 3
lo=6 : công thức gốc (ở đây n=9 thuộc trường hợp 2) = 3, DP = 3


Bài học: mỗi công thức trong bài này gắn chặt với **giả thiết cụ thể** (ngưỡng 6 đủ nhỏ để đoạn lặp không quá dài). Khi đổi đề, hãy kiểm tra lại từng giả thiết — và một trọng tài DP/BFS sẽ bắt lỗi rất nhanh.

## 23. Bài tập vận dụng

Làm theo thứ tự, từ dễ tới khó. Gợi ý ở mục 24, đáp án ở **cuối notebook**.

### Bài 1 — Đoán output (không chạy code)

Với `p = "..!!!aaaaBB1111111"`, hãy đoán:

- (a) `get_runs(p)` trả về gì?
- (b) Những đoạn nào có `L >= 3`? `L // 3` và `L % 3` của từng đoạn?
- (c) `count_missing(p)` và `count_replace(p)` bằng bao nhiêu?

### Bài 2 — Tính tay đáp án

Tính `strongPasswordChecker` cho từng mật khẩu, ghi rõ thuộc trường hợp nào và các đại lượng `missing`, `replace`, `delete`, `cnt`:

- (a) `"aaa1"`
- (b) `"ABABABABAB"`
- (c) `"aaaaaaaaBBBBBBB1234567"` (dài 22)

### Bài 3 — Điền TODO: hai hàm nền tảng

Viết lại `count_missing` và hàm trả về **danh sách độ dài** các đoạn lặp, chỉ dùng vòng lặp (không dùng `groupby`).

In [30]:
def exercise_count_missing(password: str) -> int:
    # TODO 1: tạo 3 biến has_lower, has_upper, has_digit = False
    # TODO 2: duyệt từng ký tự, bật biến tương ứng
    # TODO 3: trả về 3 - (số biến đang True)
    pass


def exercise_run_lengths(password: str) -> list:
    # TODO: hai con trỏ i, j; mỗi đoạn thêm j - i vào danh sách
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_count_missing("a") == 2
# assert exercise_count_missing("......") == 3
# assert exercise_count_missing("1337C0d3") == 0
# assert exercise_run_lengths("aaaBB1111c") == [3, 2, 4, 1]
# assert exercise_run_lengths("x") == [1]

### Bài 4 — Biến thể: trả về **chính mật khẩu mạnh** (trường hợp `6 <= n <= 20`)

Viết hàm nhận `password` có `6 <= n <= 20`, trả về một mật khẩu mạnh nhận được bằng **đúng `max(missing, replace)` lần thay** (cùng độ dài, khác bản gốc ở đúng chừng ấy vị trí).

```python
exercise_make_strong("aaaaaa")  -> ví dụ "aa1aaA"
exercise_make_strong("abcdef")  -> ví dụ "1Acdef"  (2 lần thay)
```

Tự kiểm tra bằng: `is_strong(kết_quả)` và số vị trí khác nhau `sum(x != y for x, y in zip(p, kết_quả))`.

In [31]:
def exercise_make_strong(password: str) -> str:
    # TODO 1: tính danh sách ký tự đại diện cho các loại còn thiếu, ví dụ ["A", "1"]
    # TODO 2: với mỗi đoạn lặp, thay các vị trí start+2, start+5, ...
    #         ưu tiên dùng đại diện còn thiếu; nếu hết thì dùng ký tự khác hai hàng xóm
    # TODO 3: nếu vẫn còn loại thiếu, thay thêm các vị trí "an toàn"
    pass

# Tự làm xong rồi mới mở test:
# for p in ["aaaaaa", "abcdef", "1111111111", "aaaB1bbbbbbb", "......"]:
#     q = exercise_make_strong(p)
#     assert is_strong(q) and len(q) == len(p)
#     assert sum(x != y for x, y in zip(p, q)) == max(count_missing(p), count_replace(p))

### Bài 5 — Tự viết lại toàn bộ lời giải (cách 2)

Không nhìn mục 15–16, tự viết lại hàm đầy đủ ba trường hợp, dùng mảng đếm theo mod 3.

In [32]:
def exercise_strong_password(password: str) -> int:
    # TODO: missing -> n < 6 -> (replace, cnt) -> n <= 20 -> ba giai đoạn xóa
    pass

# Tự làm xong rồi mới mở test:
# for p, expected in tests:
#     assert exercise_strong_password(p) == expected, p

### Bài 6 — Mini challenge: "Fancy string" và chỉ-được-xóa

(a) LeetCode 1957: xóa **ít ký tự nhất** để chuỗi không còn 3 ký tự giống nhau liên tiếp; trả về chuỗi kết quả.

(b) Câu hỏi tư duy: nếu chỉ được **xóa** (không chèn, không thay) và chỉ quan tâm điều kiện "không lặp 3", cần bao nhiêu lần xóa? Viết công thức theo độ dài các đoạn lặp và so với `L // 3` lần thay — vì sao xóa "đắt" hơn thay nhiều thế?

```python
exercise_fancy("leeetcode") -> "leetcode"
exercise_fancy("aaabaaaa")  -> "aabaa"
exercise_min_deletions_only("aaabaaaa") -> 3
```

In [33]:
def exercise_fancy(s: str) -> str:
    # TODO: giữ ký tự c nếu nó không tạo thành 3 ký tự giống nhau với 2 ký tự cuối đã giữ
    pass


def exercise_min_deletions_only(s: str) -> int:
    # TODO: công thức theo độ dài các đoạn lặp
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_fancy("leeetcode") == "leetcode"
# assert exercise_fancy("aaabaaaa") == "aabaa"
# assert exercise_fancy("aab") == "aab"
# assert exercise_min_deletions_only("aaabaaaa") == 3

## 24. Gợi ý theo mức

**Bài 1**
- Mức 1: đi từ trái sang phải, mỗi khi ký tự đổi là sang đoạn mới.
- Mức 2: `'.'` và `'!'` là hai ký tự khác nhau → hai đoạn riêng.
- Mức 3: có 5 đoạn; đoạn gồm toàn chữ số `'1'` dài 7.

**Bài 2**
- Mức 1: xác định trường hợp bằng `n` trước tiên.
- Mức 2: (a) `n = 4`; (b) không có đoạn lặp, chỉ còn nợ loại ký tự; (c) `delete = 2`, có hai đoạn dài 8 và 7.
- Mức 3: (c) `cnt = [0, 1, 1]`, `replace = 4`; giai đoạn 2 dùng hết 2 lần xóa cho đoạn dài 7.

**Bài 3**
- Mức 1: dùng `if c.islower(): has_lower = True` ...
- Mức 2: chép khung hai con trỏ ở mục 5, chỉ đổi `runs.append((p[i], j - i))` thành `append(j - i)`.
- Mức 3: `return 3 - (has_lower + has_upper + has_digit)`.

**Bài 4**
- Mức 1: chuyển chuỗi thành `list` để sửa từng vị trí: `s = list(password)`.
- Mức 2: ký tự mới phải khác `s[pos - 1]` và `s[pos + 1]` (nếu có); thử lần lượt trong `"aA1"` hoặc dùng đại diện đang thiếu (nó chắc chắn khác hàng xóm gốc).
- Mức 3: vị trí "an toàn" ở TODO 3 là vị trí chưa bị thay, mà ký tự gốc ở đó **không phải** đại diện duy nhất của loại của nó (đếm số ký tự mỗi loại trước).

**Bài 5**
- Mức 1: thứ tự kiểm tra: `n < 6` → tính đoạn lặp → `n <= 20` → xóa.
- Mức 2: `cnt[L % 3] += 1` chỉ với `L >= 3`.
- Mức 3: ba dòng giai đoạn: `min(rem, cnt[0])`, `min(rem, 2 * cnt[1])` với `// 2`, cuối cùng `rem // 3`.

**Bài 6**
- Mức 1: (a) dùng danh sách `out`, kiểm tra `out[-1]` và `out[-2]`.
- Mức 2: (b) mỗi đoạn dài `L` phải giữ lại tối đa 2 ký tự.
- Mức 3: (b) `Σ max(0, L - 2)`; so sánh với `L // 3`: một lần xóa chỉ "rút ngắn" 1 ký tự, một lần thay "che" được 3.

## 25. Đáp án tham khảo

Chỉ xem sau khi đã tự làm. Mỗi đáp án đều có `assert` chạy được.

### Đáp án bài 1 và bài 2

Bài 1: `get_runs` = `[('.', 2), ('!', 3), ('a', 4), ('B', 2), ('1', 7)]`. Các đoạn `L >= 3`: `!!!` (3: `//3 = 1`, `%3 = 0`), `aaaa` (4: `1`, `1`), `1111111` (7: `2`, `1`). `missing = 0`, `replace = 4`.

Bài 2:

- (a) `"aaa1"`: `n = 4 < 6`, `missing = 1` (thiếu chữ hoa), `6 - n = 2` → **2** (ví dụ chèn `A` vào giữa `aaa` và chèn thêm một ký tự: `"aaAa1b"`).
- (b) `"ABABABABAB"`: `n = 10`, không có đoạn lặp, `replace = 0`, `missing = 2` → **2**.
- (c) `"aaaaaaaaBBBBBBB1234567"`: `n = 22`, `delete = 2`, đoạn 8 (`%3 = 2`) và 7 (`%3 = 1`), `replace = 2 + 2 = 4`, `cnt = [0, 1, 1]`. GĐ1: không dùng. GĐ2: dùng 2 lần xóa cho đoạn 7 → 5, `replace = 3`. GĐ3: hết lần xóa. Đáp án `2 + max(0, 3)` = **5**.

In [34]:
p = "..!!!aaaaBB1111111"
assert get_runs(p) == [('.', 2), ('!', 3), ('a', 4), ('B', 2), ('1', 7)]
assert count_missing(p) == 0 and count_replace(p) == 4
for p, expected in [("aaa1", 2), ("ABABABABAB", 2), ("aaaaaaaaBBBBBBB1234567", 5)]:
    assert sol.strongPasswordChecker(p) == expected == strong_password_dp(p), p
trace_mod3("aaaaaaaaBBBBBBB1234567")
print("Đáp án bài 1, 2: PASS")

'aaaaaaaaBBBBBBB1234567': n=22 missing=0 replace=4 cnt=[0, 1, 1]
  GĐ1: dùng 0 lần xóa -> replace=4, rem=2
  GĐ2: dùng 2 lần xóa -> replace=3, rem=0
  GĐ3: 0 lần xóa còn lại giảm 0 -> replace=3
  đáp án = 2 + max(0, 3) = 5
Đáp án bài 1, 2: PASS


### Đáp án bài 3

In [35]:
def exercise_count_missing(password: str) -> int:
    has_lower = has_upper = has_digit = False
    for c in password:
        if c.islower():
            has_lower = True
        elif c.isupper():
            has_upper = True
        elif c.isdigit():
            has_digit = True
    return 3 - (has_lower + has_upper + has_digit)


def exercise_run_lengths(password: str) -> list:
    lengths = []
    i = 0
    while i < len(password):
        j = i
        while j < len(password) and password[j] == password[i]:
            j += 1
        lengths.append(j - i)
        i = j
    return lengths


assert exercise_count_missing("a") == 2
assert exercise_count_missing("......") == 3
assert exercise_count_missing("1337C0d3") == 0
assert exercise_run_lengths("aaaBB1111c") == [3, 2, 4, 1]
assert exercise_run_lengths("x") == [1]
print("Đáp án bài 3: PASS")

Đáp án bài 3: PASS


### Đáp án bài 4

Ba bước đúng như TODO. Vì sao ký tự đại diện của loại đang thiếu luôn khác hàng xóm? Hàng xóm bên trái có thể đã bị thay, nhưng mỗi loại chỉ có **một** đại diện và các vị trí thay trong đoạn lặp cách nhau ít nhất 3, nên đại diện không bao giờ đứng cạnh một bản sao của chính nó. Hàng xóm bên phải luôn là ký tự gốc, mà ký tự gốc không thuộc loại đang thiếu.

In [36]:
def exercise_make_strong(password: str) -> str:
    s = list(password)
    need = [rep for rep, ok in [("a", any(c.islower() for c in s)),
                                ("A", any(c.isupper() for c in s)),
                                ("1", any(c.isdigit() for c in s))] if not ok]
    changed = set()

    def pick(pos):
        if need:
            return need.pop()
        left = s[pos - 1] if pos > 0 else None
        right = s[pos + 1] if pos + 1 < len(s) else None
        return next(c for c in "aA1" if c != left and c != right)

    i = 0
    while i < len(s):                          # TODO 2: bẻ các đoạn lặp
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        for pos in range(i + 2, j, 3):
            s[pos] = pick(pos)
            changed.add(pos)
        i = j

    def cls(c):
        return 1 if c.islower() else 2 if c.isupper() else 4 if c.isdigit() else 0

    for pos in range(len(s)):                  # TODO 3: lấp nốt loại còn thiếu
        if not need:
            break
        same_class = sum(cls(c) == cls(s[pos]) for c in s)
        if pos not in changed and (cls(s[pos]) == 0 or same_class >= 2):
            s[pos] = need.pop()
            changed.add(pos)
    return "".join(s)


random.seed(11)
samples = ["aaaaaa", "abcdef", "1111111111", "aaaB1bbbbbbb", "......"]
samples += ["".join(random.choice("aaB1.") for _ in range(random.randint(6, 20))) for _ in range(300)]
for p in samples:
    q = exercise_make_strong(p)
    assert is_strong(q) and len(q) == len(p), (p, q)
    assert sum(x != y for x, y in zip(p, q)) == max(count_missing(p), count_replace(p)), (p, q)
for p in samples[:5]:
    print(f"{p!r:15} -> {exercise_make_strong(p)!r}")
print("Đáp án bài 4: PASS trên", len(samples), "chuỗi")

'aaaaaa'        -> 'aa1aaA'
'abcdef'        -> '1Acdef'
'1111111111'    -> '11A11a11a1'
'aaaB1bbbbbbb'  -> 'aaAB1bbabbab'
'......'        -> 'a.1..A'
Đáp án bài 4: PASS trên 305 chuỗi


### Đáp án bài 5

In [37]:
def exercise_strong_password(password: str) -> int:
    n = len(password)
    missing = 3 - (any(c.islower() for c in password)
                   + any(c.isupper() for c in password)
                   + any(c.isdigit() for c in password))
    if n < 6:
        return max(missing, 6 - n)
    replace, cnt = 0, [0, 0, 0]
    i = 0
    while i < n:
        j = i
        while j < n and password[j] == password[i]:
            j += 1
        if j - i >= 3:
            replace += (j - i) // 3
            cnt[(j - i) % 3] += 1
        i = j
    if n <= 20:
        return max(missing, replace)
    delete = rem = n - 20
    use = min(rem, cnt[0]); replace -= use; rem -= use
    use = min(rem, 2 * cnt[1]); replace -= use // 2; rem -= use
    replace = max(0, replace - rem // 3)
    return delete + max(missing, replace)


for p, expected in tests:
    assert exercise_strong_password(p) == expected, p
random.seed(5)
for _ in range(500):
    p = random_runny_password(random.randint(1, 50))
    assert exercise_strong_password(p) == strong_password_greedy(p), p
print("Đáp án bài 5: PASS")

Đáp án bài 5: PASS


### Đáp án bài 6

(b) Chỉ được xóa thì mỗi đoạn dài `L` phải giảm về 2 → `Σ max(0, L - 2)` lần. Với đoạn dài 9: xóa cần 7 lần, thay chỉ cần 3. Một lần xóa chỉ bớt đúng 1 ký tự, còn một lần thay "dựng rào" cho cả khúc 3 ký tự — đó là lý do lời giải chính luôn ưu tiên thay, và chỉ xóa khi **bị bắt buộc** (`n > 20`).

In [38]:
def exercise_fancy(s: str) -> str:
    out = []
    for c in s:
        if len(out) >= 2 and out[-1] == c and out[-2] == c:
            continue                       # giữ c sẽ tạo 3 ký tự lặp: bỏ
        out.append(c)
    return "".join(out)


def exercise_min_deletions_only(s: str) -> int:
    return sum(max(0, L - 2) for ch, L in get_runs(s))


assert exercise_fancy("leeetcode") == "leetcode"
assert exercise_fancy("aaabaaaa") == "aabaa"
assert exercise_fancy("aab") == "aab"
assert exercise_min_deletions_only("aaabaaaa") == 3
random.seed(3)
for _ in range(300):
    s = random_runny_password(random.randint(1, 30))
    f = exercise_fancy(s)
    assert not has_triple(f) and len(s) - len(f) == exercise_min_deletions_only(s)
print("Đáp án bài 6: PASS")

Đáp án bài 6: PASS


## 26. Tổng kết: template dễ nhớ

```text
missing = 3 - (có thường + có hoa + có số)
n < 6      → max(missing, 6 - n)                       # chèn đủ, đoạn lặp tự gãy
replace = Σ L//3,  cnt[L%3] += 1   (mọi đoạn lặp L >= 3)
n <= 20    → max(missing, replace)                     # thay vừa bẻ lặp vừa thêm loại
n > 20     → delete = n - 20, rem = delete
             GĐ1: %3==0, giá 1   → replace -= min(rem, cnt[0])
             GĐ2: %3==1, giá 2   → replace -= min(rem, 2*cnt[1]) // 2
             GĐ3: còn lại giá 3  → replace -= rem // 3   (không âm)
             return delete + max(missing, replace)
```

Câu thần chú: **"Thiếu thì chèn, vừa thì thay, thừa thì xóa — xóa ưu tiên `%3 == 0`, rồi `%3 == 1`, rồi chia 3."**

## Checklist trước khi submit

- [ ] `missing` tính đúng: `.` và `!` **không** thuộc loại nào.
- [ ] Trường hợp `n < 6` dùng `max`, không cộng; không cần xét đoạn lặp.
- [ ] Trường hợp `6..20` dùng `max(missing, replace)`, không cộng.
- [ ] Chỉ đếm đoạn lặp có `L >= 3`; `cnt` theo `L % 3`.
- [ ] Giai đoạn 2 dùng `2 * cnt[1]` và `use // 2`; giai đoạn 3 không để `replace` âm.
- [ ] Đáp án `n > 20` **cộng thêm `delete`**.
- [ ] Đã test: 3 ví dụ đề, `"aaaaa"`, `"aaaaaa"`, chuỗi 21 ký tự có đoạn `%3 == 0`, `"a" * 25`, `"." * 50`.
- [ ] Đã đối chiếu với trọng tài (BFS cho chuỗi ngắn, DP cho chuỗi dài) trên input ngẫu nhiên.